# ML Metadata Lab: Breast Cancer Classification Pipeline

This lab demonstrates how to use [ML Metadata (MLMD)](https://www.tensorflow.org/tfx/guide/mlmd) to track every stage of a real multi-step ML pipeline — from raw data ingestion through model evaluation.

Unlike the simple schema-only walkthrough, this lab builds a **complete four-step pipeline** with full lineage tracking:

| Step | Component | Inputs | Outputs |
|------|-----------|--------|---------|
| 1 | Data Validation | Raw CSV splits | Schema |
| 2 | Feature Engineering | Raw data + Schema | Scaled CSV splits + Scaler |
| 3 | Model Training | Preprocessed train data | Trained model file |
| 4 | Model Evaluation | Trained model + Preprocessed eval data | Metrics JSON |

We run Steps 3 & 4 **twice** — once with LogisticRegression (v1) and once with RandomForestClassifier (v2) — and use MLMD to compare experiments. We also record a **failed training attempt** to show real-world recovery tracking.

**Dataset:** [Breast Cancer Wisconsin](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) — 569 samples, 30 numeric features, binary classification (malignant vs. benign).

**Storage:** Persistent SQLite database (`./mlmd.sqlite`) — metadata survives between notebook runs.

## Imports

In [ ]:
from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2

import tensorflow as tf
print('TF version: {}'.format(tf.__version__))

import tensorflow_data_validation as tfdv
print('TFDV version: {}'.format(tfdv.version.__version__))

import pandas as pd
import numpy as np
import os
import json
import uuid
import joblib
import warnings

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.exceptions import ConvergenceWarning

print('sklearn imported successfully')

## Prepare the Dataset

We load the Breast Cancer Wisconsin dataset from `sklearn` and split it into three CSV files that mirror the standard TFX pipeline layout:

- `data/train/data.csv` — ~398 rows used for training
- `data/eval/data.csv` — ~114 rows used for evaluation
- `data/serving/data.csv` — ~57 rows (no target column) used for serving

Each feature is a real-valued measurement of a digitized image of a breast mass (e.g., radius, texture, perimeter, area, smoothness). The target is 0 = malignant, 1 = benign.

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']

# 70 / 20 / 10 stratified split
X_temp, X_serving, y_temp, y_serving = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_eval, y_train, y_eval = train_test_split(
    X_temp, y_temp, test_size=0.2222, random_state=42, stratify=y_temp)

train_df   = X_train.copy(); train_df['target']   = y_train.values
eval_df    = X_eval.copy();  eval_df['target']    = y_eval.values
serving_df = X_serving.copy()   # no target

os.makedirs('./data/train',   exist_ok=True)
os.makedirs('./data/eval',    exist_ok=True)
os.makedirs('./data/serving', exist_ok=True)

train_df.reset_index(drop=True).to_csv('./data/train/data.csv',   index=False)
eval_df.reset_index(drop=True).to_csv('./data/eval/data.csv',     index=False)
serving_df.reset_index(drop=True).to_csv('./data/serving/data.csv', index=False)

print('Dataset splits saved:')
print('  train/data.csv  :', len(train_df), 'rows')
print('  eval/data.csv   :', len(eval_df), 'rows')
print('  serving/data.csv:', len(serving_df), 'rows (no target column)')
print('Features:', list(data.feature_names))
print('Classes :', list(data.target_names), '(0=malignant, 1=benign)')

## MLMD Data Model Overview

<img src='img/mlmd_overview.png' alt='MLMD overview diagram'>

The core MLMD components we will use:

| Component | Role | Our usage |
|-----------|------|-----------|
| **ArtifactType** | Schema for artifact metadata | DataSet, Schema, PreprocessedData, TrainedModel, EvaluationMetrics |
| **Artifact** | One concrete file / artifact instance | Each CSV, schema file, model file, metrics JSON |
| **ExecutionType** | Schema for execution metadata | Data Validation, Feature Engineering, Model Training, Model Evaluation |
| **Execution** | One run of a pipeline component | Each TFDV run, scaler fit, model.fit(), model.predict() |
| **Event** | Artifact ↔ Execution edge | DECLARED_INPUT / DECLARED_OUTPUT |
| **ContextType** | Schema for grouping contexts | Pipeline, Experiment |
| **Context** | One group (pipeline run / experiment) | One pipeline run, two experiment variants |
| **Attribution** | Artifact ↔ Context edge | Links artifacts into contexts |
| **Association** | Execution ↔ Context edge | Links executions into contexts |

## Step 1 — Define the Metadata Store

We use a **persistent SQLite database** instead of the fake in-memory store. This means metadata survives between notebook sessions and can be inspected with any SQLite browser.

In [ ]:
connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.sqlite.filename_uri = './mlmd.sqlite'
connection_config.sqlite.connection_mode = 3  # READWRITE_OPENCREATE

store = metadata_store.MetadataStore(connection_config)
print('Persistent metadata store ready at: ./mlmd.sqlite')

## Step 2 — Register Artifact Types

We register five artifact types — one for each kind of file our pipeline produces. Note the new types compared to the basic lab: `PreprocessedData`, `TrainedModel`, and `EvaluationMetrics`.

In [ ]:
# --- 1. DataSet: raw CSV input splits ---
data_artifact_type = metadata_store_pb2.ArtifactType()
data_artifact_type.name = 'DataSet'
data_artifact_type.properties['name']     = metadata_store_pb2.STRING
data_artifact_type.properties['split']    = metadata_store_pb2.STRING
data_artifact_type.properties['version']  = metadata_store_pb2.INT
data_artifact_type.properties['num_rows'] = metadata_store_pb2.INT
data_artifact_type_id = store.put_artifact_type(data_artifact_type)

# --- 2. Schema: TFDV-generated .pbtxt schema file ---
schema_artifact_type = metadata_store_pb2.ArtifactType()
schema_artifact_type.name = 'Schema'
schema_artifact_type.properties['name']    = metadata_store_pb2.STRING
schema_artifact_type.properties['version'] = metadata_store_pb2.INT
schema_artifact_type_id = store.put_artifact_type(schema_artifact_type)

# --- 3. PreprocessedData: StandardScaler-transformed CSV ---
preprocessed_artifact_type = metadata_store_pb2.ArtifactType()
preprocessed_artifact_type.name = 'PreprocessedData'
preprocessed_artifact_type.properties['name']        = metadata_store_pb2.STRING
preprocessed_artifact_type.properties['split']       = metadata_store_pb2.STRING
preprocessed_artifact_type.properties['scaler_path'] = metadata_store_pb2.STRING
preprocessed_artifact_type_id = store.put_artifact_type(preprocessed_artifact_type)

# --- 4. TrainedModel: serialised sklearn model (.joblib) ---
model_artifact_type = metadata_store_pb2.ArtifactType()
model_artifact_type.name = 'TrainedModel'
model_artifact_type.properties['name']       = metadata_store_pb2.STRING
model_artifact_type.properties['model_type'] = metadata_store_pb2.STRING
model_artifact_type.properties['version']    = metadata_store_pb2.INT
model_artifact_type_id = store.put_artifact_type(model_artifact_type)

# --- 5. EvaluationMetrics: JSON file with classification metrics ---
metrics_artifact_type = metadata_store_pb2.ArtifactType()
metrics_artifact_type.name = 'EvaluationMetrics'
metrics_artifact_type.properties['name']          = metadata_store_pb2.STRING
metrics_artifact_type.properties['model_version'] = metadata_store_pb2.INT
metrics_artifact_type.properties['accuracy']      = metadata_store_pb2.DOUBLE
metrics_artifact_type.properties['f1_score']      = metadata_store_pb2.DOUBLE
metrics_artifact_type_id = store.put_artifact_type(metrics_artifact_type)

print('Registered artifact types:')
for at in store.get_artifact_types():
    print('  [{}] {}'.format(at.id, at.name))

## Step 3 — Register Execution Types

Each pipeline component gets its own `ExecutionType`. We store `state` on all of them (RUNNING / COMPLETED / FAILED) and add component-specific properties like `scaler`, `model_type`, and `hyperparameters`.

In [ ]:
# --- 1. Data Validation ---
dv_exec_type = metadata_store_pb2.ExecutionType()
dv_exec_type.name = 'Data Validation'
dv_exec_type.properties['state'] = metadata_store_pb2.STRING
dv_exec_type_id = store.put_execution_type(dv_exec_type)

# --- 2. Feature Engineering ---
fe_exec_type = metadata_store_pb2.ExecutionType()
fe_exec_type.name = 'Feature Engineering'
fe_exec_type.properties['state']  = metadata_store_pb2.STRING
fe_exec_type.properties['scaler'] = metadata_store_pb2.STRING
fe_exec_type_id = store.put_execution_type(fe_exec_type)

# --- 3. Model Training ---
mt_exec_type = metadata_store_pb2.ExecutionType()
mt_exec_type.name = 'Model Training'
mt_exec_type.properties['state']            = metadata_store_pb2.STRING
mt_exec_type.properties['model_type']       = metadata_store_pb2.STRING
mt_exec_type.properties['hyperparameters']  = metadata_store_pb2.STRING
mt_exec_type_id = store.put_execution_type(mt_exec_type)

# --- 4. Model Evaluation ---
me_exec_type = metadata_store_pb2.ExecutionType()
me_exec_type.name = 'Model Evaluation'
me_exec_type.properties['state'] = metadata_store_pb2.STRING
me_exec_type_id = store.put_execution_type(me_exec_type)

print('Registered execution types:')
for et in store.get_execution_types():
    print('  [{}] {}'.format(et.id, et.name))

## Step 4 — Register Context Types

We create two context types:
- **Pipeline**: groups everything that ran in one full pipeline execution
- **Experiment**: groups a specific model variant (LogReg or RandomForest) including its training and evaluation

In [ ]:
# --- Pipeline context type ---
pipeline_context_type = metadata_store_pb2.ContextType()
pipeline_context_type.name = 'Pipeline'
pipeline_context_type.properties['pipeline_name'] = metadata_store_pb2.STRING
pipeline_context_type.properties['run_id']        = metadata_store_pb2.STRING
pipeline_context_type_id = store.put_context_type(pipeline_context_type)

# --- Experiment context type ---
expt_context_type = metadata_store_pb2.ContextType()
expt_context_type.name = 'Experiment'
expt_context_type.properties['model_type'] = metadata_store_pb2.STRING
expt_context_type.properties['note']       = metadata_store_pb2.STRING
expt_context_type_id = store.put_context_type(expt_context_type)

print('Registered context types:')
for ct in store.get_context_types():
    print('  [{}] {}'.format(ct.id, ct.name))

---
## Pipeline Step 1 — Data Validation

We run TFDV on both the **train** and **eval** splits:
1. Register both raw dataset artifacts as inputs
2. Run `generate_statistics_from_csv` on each split
3. Infer a schema from the training statistics
4. Validate eval statistics against the inferred schema (checks for anomalies)
5. Record the schema as the output artifact

In [ ]:
os.makedirs('./artifacts/preprocessed', exist_ok=True)
os.makedirs('./artifacts/models',       exist_ok=True)
os.makedirs('./artifacts/metrics',      exist_ok=True)

# ── Input Artifacts ────────────────────────────────────────────────────────
train_artifact = metadata_store_pb2.Artifact()
train_artifact.uri = './data/train/data.csv'
train_artifact.type_id = data_artifact_type_id
train_artifact.properties['name'].string_value    = 'Breast Cancer Dataset'
train_artifact.properties['split'].string_value   = 'train'
train_artifact.properties['version'].int_value    = 1
train_artifact.properties['num_rows'].int_value   = len(train_df)
train_artifact_id = store.put_artifacts([train_artifact])[0]

eval_artifact = metadata_store_pb2.Artifact()
eval_artifact.uri = './data/eval/data.csv'
eval_artifact.type_id = data_artifact_type_id
eval_artifact.properties['name'].string_value    = 'Breast Cancer Dataset'
eval_artifact.properties['split'].string_value   = 'eval'
eval_artifact.properties['version'].int_value    = 1
eval_artifact.properties['num_rows'].int_value   = len(eval_df)
eval_artifact_id = store.put_artifacts([eval_artifact])[0]

print('Input artifacts registered  | train ID:', train_artifact_id, ' eval ID:', eval_artifact_id)

# ── Execution ──────────────────────────────────────────────────────────────
dv_execution = metadata_store_pb2.Execution()
dv_execution.type_id = dv_exec_type_id
dv_execution.properties['state'].string_value = 'RUNNING'
dv_execution_id = store.put_executions([dv_execution])[0]

# ── Input Events ───────────────────────────────────────────────────────────
for art_id in [train_artifact_id, eval_artifact_id]:
    ev = metadata_store_pb2.Event()
    ev.artifact_id  = art_id
    ev.execution_id = dv_execution_id
    ev.type = metadata_store_pb2.Event.DECLARED_INPUT
    store.put_events([ev])

# ── Run TFDV ───────────────────────────────────────────────────────────────
print('Generating statistics...')
train_stats = tfdv.generate_statistics_from_csv('./data/train/data.csv')
eval_stats  = tfdv.generate_statistics_from_csv('./data/eval/data.csv')

schema = tfdv.infer_schema(statistics=train_stats)
schema_file = './artifacts/schema.pbtxt'
tfdv.write_schema_text(schema, schema_file)

# Validate eval statistics against the schema
anomalies = tfdv.validate_statistics(eval_stats, schema)
print('Schema written to:', schema_file)
if str(anomalies.anomaly_info):
    print('Eval anomalies found:', anomalies)
else:
    print('Eval split: no schema anomalies detected.')

# ── Output Artifact ────────────────────────────────────────────────────────
schema_artifact = metadata_store_pb2.Artifact()
schema_artifact.uri = schema_file
schema_artifact.type_id = schema_artifact_type_id
schema_artifact.properties['name'].string_value    = 'Breast Cancer Schema'
schema_artifact.properties['version'].int_value    = 1
schema_artifact_id = store.put_artifacts([schema_artifact])[0]

# ── Output Event ───────────────────────────────────────────────────────────
out_ev = metadata_store_pb2.Event()
out_ev.artifact_id  = schema_artifact_id
out_ev.execution_id = dv_execution_id
out_ev.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([out_ev])

# ── Update execution state ─────────────────────────────────────────────────
dv_execution.id = dv_execution_id
dv_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([dv_execution])

print('\nData Validation execution ID:', dv_execution_id, '| state: COMPLETED')
print('Schema artifact ID:', schema_artifact_id)

---
## Pipeline Step 2 — Feature Engineering

We fit a `StandardScaler` on the **training** split only (to prevent data leakage) and then transform both train and eval splits. The scaler object is saved to disk alongside the preprocessed CSVs so it can be applied to serving data later.

In [ ]:
# ── Execution ──────────────────────────────────────────────────────────────
fe_execution = metadata_store_pb2.Execution()
fe_execution.type_id = fe_exec_type_id
fe_execution.properties['state'].string_value  = 'RUNNING'
fe_execution.properties['scaler'].string_value = 'StandardScaler'
fe_execution_id = store.put_executions([fe_execution])[0]

# ── Input Events (raw datasets + schema) ───────────────────────────────────
for art_id in [train_artifact_id, eval_artifact_id, schema_artifact_id]:
    ev = metadata_store_pb2.Event()
    ev.artifact_id  = art_id
    ev.execution_id = fe_execution_id
    ev.type = metadata_store_pb2.Event.DECLARED_INPUT
    store.put_events([ev])

# ── Run Feature Engineering ────────────────────────────────────────────────
feature_cols = [c for c in train_df.columns if c != 'target']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[feature_cols])
X_eval_scaled  = scaler.transform(eval_df[feature_cols])

train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
train_scaled_df['target'] = train_df['target'].values

eval_scaled_df = pd.DataFrame(X_eval_scaled, columns=feature_cols)
eval_scaled_df['target'] = eval_df['target'].values

train_scaled_df.to_csv('./artifacts/preprocessed/train.csv', index=False)
eval_scaled_df.to_csv('./artifacts/preprocessed/eval.csv',   index=False)

scaler_path = './artifacts/preprocessed/scaler.joblib'
joblib.dump(scaler, scaler_path)

print('Preprocessed splits saved.')
print('  train shape:', train_scaled_df.shape)
print('  eval  shape:', eval_scaled_df.shape)
print('  scaler saved to:', scaler_path)

# ── Output Artifacts ───────────────────────────────────────────────────────
train_prep_artifact = metadata_store_pb2.Artifact()
train_prep_artifact.uri = './artifacts/preprocessed/train.csv'
train_prep_artifact.type_id = preprocessed_artifact_type_id
train_prep_artifact.properties['name'].string_value        = 'Breast Cancer Preprocessed'
train_prep_artifact.properties['split'].string_value       = 'train'
train_prep_artifact.properties['scaler_path'].string_value = scaler_path
train_prep_artifact_id = store.put_artifacts([train_prep_artifact])[0]

eval_prep_artifact = metadata_store_pb2.Artifact()
eval_prep_artifact.uri = './artifacts/preprocessed/eval.csv'
eval_prep_artifact.type_id = preprocessed_artifact_type_id
eval_prep_artifact.properties['name'].string_value        = 'Breast Cancer Preprocessed'
eval_prep_artifact.properties['split'].string_value       = 'eval'
eval_prep_artifact.properties['scaler_path'].string_value = scaler_path
eval_prep_artifact_id = store.put_artifacts([eval_prep_artifact])[0]

# ── Output Events ──────────────────────────────────────────────────────────
for art_id in [train_prep_artifact_id, eval_prep_artifact_id]:
    ev = metadata_store_pb2.Event()
    ev.artifact_id  = art_id
    ev.execution_id = fe_execution_id
    ev.type = metadata_store_pb2.Event.DECLARED_OUTPUT
    store.put_events([ev])

# ── Update execution state ─────────────────────────────────────────────────
fe_execution.id = fe_execution_id
fe_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([fe_execution])

print('\nFeature Engineering execution ID:', fe_execution_id, '| state: COMPLETED')
print('Preprocessed train artifact ID:', train_prep_artifact_id)
print('Preprocessed eval  artifact ID:', eval_prep_artifact_id)

---
## Pipeline Step 3a — Simulated FAILED Training Attempt

In production, training runs don't always succeed. Here we deliberately trigger a `ConvergenceWarning` by setting `max_iter=1` — far too few iterations for the solver to converge. We catch the warning as an error and record the execution state as **FAILED** in MLMD.

This is a realistic pattern: MLMD lets you audit not just what succeeded, but also what failed and why.

In [ ]:
# ── Execution (intentionally bad hyperparams) ──────────────────────────────
failed_execution = metadata_store_pb2.Execution()
failed_execution.type_id = mt_exec_type_id
failed_execution.properties['state'].string_value           = 'RUNNING'
failed_execution.properties['model_type'].string_value      = 'LogisticRegression'
failed_execution.properties['hyperparameters'].string_value = json.dumps({'max_iter': 1, 'C': 1.0})
failed_execution_id = store.put_executions([failed_execution])[0]

# Input event
ev = metadata_store_pb2.Event()
ev.artifact_id  = train_prep_artifact_id
ev.execution_id = failed_execution_id
ev.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([ev])

# ── Attempt training — expect failure ─────────────────────────────────────
X_tr = train_scaled_df.drop('target', axis=1).values
y_tr = train_scaled_df['target'].values

try:
    with warnings.catch_warnings():
        warnings.simplefilter('error', ConvergenceWarning)
        bad_model = LogisticRegression(max_iter=1, C=1.0, random_state=42)
        bad_model.fit(X_tr, y_tr)
    print('Model converged unexpectedly — no failure recorded.')
except (ConvergenceWarning, Exception) as e:
    failed_execution.id = failed_execution_id
    failed_execution.properties['state'].string_value = 'FAILED'
    store.put_executions([failed_execution])
    print('Training FAILED as expected:', type(e).__name__)
    print('Execution ID', failed_execution_id, 'recorded as FAILED in metadata store.')

---
## Pipeline Step 3b — Model Training v1 (LogisticRegression)

After correcting the hyperparameters we run a proper training execution. The fitted model is serialised with `joblib` and registered as a `TrainedModel` artifact.

In [ ]:
hp_v1 = {'max_iter': 1000, 'C': 1.0, 'solver': 'lbfgs'}

# ── Execution ──────────────────────────────────────────────────────────────
mt_v1_execution = metadata_store_pb2.Execution()
mt_v1_execution.type_id = mt_exec_type_id
mt_v1_execution.properties['state'].string_value           = 'RUNNING'
mt_v1_execution.properties['model_type'].string_value      = 'LogisticRegression'
mt_v1_execution.properties['hyperparameters'].string_value = json.dumps(hp_v1)
mt_v1_execution_id = store.put_executions([mt_v1_execution])[0]

# Input event
ev = metadata_store_pb2.Event()
ev.artifact_id  = train_prep_artifact_id
ev.execution_id = mt_v1_execution_id
ev.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([ev])

# ── Train ──────────────────────────────────────────────────────────────────
lr_model = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42)
lr_model.fit(X_tr, y_tr)

model_v1_path = './artifacts/models/logreg_v1.joblib'
joblib.dump(lr_model, model_v1_path)
print('LogisticRegression model saved to', model_v1_path)

# ── Output Artifact ────────────────────────────────────────────────────────
model_v1_artifact = metadata_store_pb2.Artifact()
model_v1_artifact.uri = model_v1_path
model_v1_artifact.type_id = model_artifact_type_id
model_v1_artifact.properties['name'].string_value       = 'Breast Cancer Classifier v1'
model_v1_artifact.properties['model_type'].string_value = 'LogisticRegression'
model_v1_artifact.properties['version'].int_value       = 1
model_v1_artifact_id = store.put_artifacts([model_v1_artifact])[0]

# Output event
ev = metadata_store_pb2.Event()
ev.artifact_id  = model_v1_artifact_id
ev.execution_id = mt_v1_execution_id
ev.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([ev])

# Update execution
mt_v1_execution.id = mt_v1_execution_id
mt_v1_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([mt_v1_execution])

print('Model Training v1 execution ID:', mt_v1_execution_id, '| state: COMPLETED')
print('TrainedModel v1 artifact ID:', model_v1_artifact_id)

---
## Pipeline Step 4a — Model Evaluation v1

We evaluate the LogisticRegression model on the preprocessed eval split and record accuracy, precision, recall, and F1 score. The full metrics dictionary is saved to a JSON file and registered as an `EvaluationMetrics` artifact. Note that `accuracy` and `f1_score` are also stored directly as artifact properties so they can be compared across experiments without loading the file.

In [ ]:
X_ev = eval_scaled_df.drop('target', axis=1).values
y_ev = eval_scaled_df['target'].values

# ── Execution ──────────────────────────────────────────────────────────────
me_v1_execution = metadata_store_pb2.Execution()
me_v1_execution.type_id = me_exec_type_id
me_v1_execution.properties['state'].string_value = 'RUNNING'
me_v1_execution_id = store.put_executions([me_v1_execution])[0]

# Input events: model + eval preprocessed data
for art_id in [model_v1_artifact_id, eval_prep_artifact_id]:
    ev = metadata_store_pb2.Event()
    ev.artifact_id  = art_id
    ev.execution_id = me_v1_execution_id
    ev.type = metadata_store_pb2.Event.DECLARED_INPUT
    store.put_events([ev])

# ── Evaluate ───────────────────────────────────────────────────────────────
y_pred_v1 = lr_model.predict(X_ev)

metrics_v1 = {
    'model_type' : 'LogisticRegression',
    'version'    : 1,
    'accuracy'   : float(accuracy_score(y_ev, y_pred_v1)),
    'precision'  : float(precision_score(y_ev, y_pred_v1)),
    'recall'     : float(recall_score(y_ev, y_pred_v1)),
    'f1_score'   : float(f1_score(y_ev, y_pred_v1)),
}

metrics_v1_path = './artifacts/metrics/metrics_v1.json'
with open(metrics_v1_path, 'w') as f:
    json.dump(metrics_v1, f, indent=2)

print('Evaluation Metrics v1 (LogisticRegression):')
for k, v in metrics_v1.items():
    print('  {:<12}: {}'.format(k, v))

# ── Output Artifact ────────────────────────────────────────────────────────
metrics_v1_artifact = metadata_store_pb2.Artifact()
metrics_v1_artifact.uri = metrics_v1_path
metrics_v1_artifact.type_id = metrics_artifact_type_id
metrics_v1_artifact.properties['name'].string_value          = 'Eval Metrics v1 - LogReg'
metrics_v1_artifact.properties['model_version'].int_value    = 1
metrics_v1_artifact.properties['accuracy'].double_value      = metrics_v1['accuracy']
metrics_v1_artifact.properties['f1_score'].double_value      = metrics_v1['f1_score']
metrics_v1_artifact_id = store.put_artifacts([metrics_v1_artifact])[0]

# Output event
ev = metadata_store_pb2.Event()
ev.artifact_id  = metrics_v1_artifact_id
ev.execution_id = me_v1_execution_id
ev.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([ev])

# Update execution
me_v1_execution.id = me_v1_execution_id
me_v1_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([me_v1_execution])

print('\nModel Evaluation v1 execution ID:', me_v1_execution_id, '| state: COMPLETED')
print('EvaluationMetrics v1 artifact ID:', metrics_v1_artifact_id)

---
## Pipeline Step 3c — Model Training v2 (RandomForestClassifier)

We re-use the same preprocessed training data to train a second model. This is the core versioning use-case for MLMD: the input artifacts are identical, but the execution type, hyperparameters, and output artifacts differ — making it straightforward to compare.

In [ ]:
hp_v2 = {'n_estimators': 100, 'max_depth': 10, 'random_state': 42}

# ── Execution ──────────────────────────────────────────────────────────────
mt_v2_execution = metadata_store_pb2.Execution()
mt_v2_execution.type_id = mt_exec_type_id
mt_v2_execution.properties['state'].string_value           = 'RUNNING'
mt_v2_execution.properties['model_type'].string_value      = 'RandomForestClassifier'
mt_v2_execution.properties['hyperparameters'].string_value = json.dumps(hp_v2)
mt_v2_execution_id = store.put_executions([mt_v2_execution])[0]

# Input event
ev = metadata_store_pb2.Event()
ev.artifact_id  = train_prep_artifact_id
ev.execution_id = mt_v2_execution_id
ev.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([ev])

# ── Train ──────────────────────────────────────────────────────────────────
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_tr, y_tr)

model_v2_path = './artifacts/models/rf_v2.joblib'
joblib.dump(rf_model, model_v2_path)
print('RandomForestClassifier model saved to', model_v2_path)

# ── Output Artifact ────────────────────────────────────────────────────────
model_v2_artifact = metadata_store_pb2.Artifact()
model_v2_artifact.uri = model_v2_path
model_v2_artifact.type_id = model_artifact_type_id
model_v2_artifact.properties['name'].string_value       = 'Breast Cancer Classifier v2'
model_v2_artifact.properties['model_type'].string_value = 'RandomForestClassifier'
model_v2_artifact.properties['version'].int_value       = 2
model_v2_artifact_id = store.put_artifacts([model_v2_artifact])[0]

# Output event
ev = metadata_store_pb2.Event()
ev.artifact_id  = model_v2_artifact_id
ev.execution_id = mt_v2_execution_id
ev.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([ev])

# Update execution
mt_v2_execution.id = mt_v2_execution_id
mt_v2_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([mt_v2_execution])

print('Model Training v2 execution ID:', mt_v2_execution_id, '| state: COMPLETED')
print('TrainedModel v2 artifact ID:', model_v2_artifact_id)

---
## Pipeline Step 4b — Model Evaluation v2

In [ ]:
# ── Execution ──────────────────────────────────────────────────────────────
me_v2_execution = metadata_store_pb2.Execution()
me_v2_execution.type_id = me_exec_type_id
me_v2_execution.properties['state'].string_value = 'RUNNING'
me_v2_execution_id = store.put_executions([me_v2_execution])[0]

# Input events: model v2 + same eval preprocessed data
for art_id in [model_v2_artifact_id, eval_prep_artifact_id]:
    ev = metadata_store_pb2.Event()
    ev.artifact_id  = art_id
    ev.execution_id = me_v2_execution_id
    ev.type = metadata_store_pb2.Event.DECLARED_INPUT
    store.put_events([ev])

# ── Evaluate ───────────────────────────────────────────────────────────────
y_pred_v2 = rf_model.predict(X_ev)

metrics_v2 = {
    'model_type' : 'RandomForestClassifier',
    'version'    : 2,
    'accuracy'   : float(accuracy_score(y_ev, y_pred_v2)),
    'precision'  : float(precision_score(y_ev, y_pred_v2)),
    'recall'     : float(recall_score(y_ev, y_pred_v2)),
    'f1_score'   : float(f1_score(y_ev, y_pred_v2)),
}

metrics_v2_path = './artifacts/metrics/metrics_v2.json'
with open(metrics_v2_path, 'w') as f:
    json.dump(metrics_v2, f, indent=2)

print('Evaluation Metrics v2 (RandomForestClassifier):')
for k, v in metrics_v2.items():
    print('  {:<12}: {}'.format(k, v))

# ── Output Artifact ────────────────────────────────────────────────────────
metrics_v2_artifact = metadata_store_pb2.Artifact()
metrics_v2_artifact.uri = metrics_v2_path
metrics_v2_artifact.type_id = metrics_artifact_type_id
metrics_v2_artifact.properties['name'].string_value          = 'Eval Metrics v2 - RandomForest'
metrics_v2_artifact.properties['model_version'].int_value    = 2
metrics_v2_artifact.properties['accuracy'].double_value      = metrics_v2['accuracy']
metrics_v2_artifact.properties['f1_score'].double_value      = metrics_v2['f1_score']
metrics_v2_artifact_id = store.put_artifacts([metrics_v2_artifact])[0]

# Output event
ev = metadata_store_pb2.Event()
ev.artifact_id  = metrics_v2_artifact_id
ev.execution_id = me_v2_execution_id
ev.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([ev])

# Update execution
me_v2_execution.id = me_v2_execution_id
me_v2_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([me_v2_execution])

print('\nModel Evaluation v2 execution ID:', me_v2_execution_id, '| state: COMPLETED')
print('EvaluationMetrics v2 artifact ID:', metrics_v2_artifact_id)

---
## Step 5 — Contexts: Grouping the Pipeline Run and Experiments

### 5a — Pipeline Context

A `Pipeline` context groups **every** artifact and execution from this entire run. This is the top-level grouping that lets you answer "what ran together?"

In [ ]:
# ── Create Pipeline context ────────────────────────────────────────────────
pipeline_context = metadata_store_pb2.Context()
pipeline_context.type_id = pipeline_context_type_id
pipeline_context.name = 'breast-cancer-pipeline-run-1'
pipeline_context.properties['pipeline_name'].string_value = 'Breast Cancer Classification Pipeline'
pipeline_context.properties['run_id'].string_value        = str(uuid.uuid4())[:8]
pipeline_context_id = store.put_contexts([pipeline_context])[0]

print('Pipeline context ID:', pipeline_context_id)
print('Pipeline run_id    :', pipeline_context.properties['run_id'].string_value)

# ── Associate ALL executions ───────────────────────────────────────────────
all_execution_ids = [
    dv_execution_id, fe_execution_id,
    failed_execution_id,
    mt_v1_execution_id, me_v1_execution_id,
    mt_v2_execution_id, me_v2_execution_id,
]

associations = []
for exec_id in all_execution_ids:
    assoc = metadata_store_pb2.Association()
    assoc.execution_id = exec_id
    assoc.context_id   = pipeline_context_id
    associations.append(assoc)

# ── Attribute ALL artifacts ────────────────────────────────────────────────
all_artifact_ids = [
    train_artifact_id, eval_artifact_id,
    schema_artifact_id,
    train_prep_artifact_id, eval_prep_artifact_id,
    model_v1_artifact_id,   metrics_v1_artifact_id,
    model_v2_artifact_id,   metrics_v2_artifact_id,
]

attributions = []
for art_id in all_artifact_ids:
    attr = metadata_store_pb2.Attribution()
    attr.artifact_id = art_id
    attr.context_id  = pipeline_context_id
    attributions.append(attr)

store.put_attributions_and_associations(attributions, associations)
print('Linked {} executions and {} artifacts to Pipeline context.'.format(
    len(associations), len(attributions)))

### 5b — Experiment Contexts

Each model variant gets its own `Experiment` context scoped to its training + evaluation steps and outputs. This lets us query "what artifacts belong to the LogReg experiment?" independently of the broader pipeline.

In [ ]:
# ── Experiment v1: LogisticRegression ─────────────────────────────────────
expt_v1_context = metadata_store_pb2.Context()
expt_v1_context.type_id = expt_context_type_id
expt_v1_context.name = 'experiment-logreg-v1'
expt_v1_context.properties['model_type'].string_value = 'LogisticRegression'
expt_v1_context.properties['note'].string_value       = 'LR C=1.0, max_iter=1000, solver=lbfgs'
expt_v1_context_id = store.put_contexts([expt_v1_context])[0]

store.put_attributions_and_associations(
    [
        metadata_store_pb2.Attribution(artifact_id=model_v1_artifact_id,   context_id=expt_v1_context_id),
        metadata_store_pb2.Attribution(artifact_id=metrics_v1_artifact_id, context_id=expt_v1_context_id),
    ],
    [
        metadata_store_pb2.Association(execution_id=mt_v1_execution_id, context_id=expt_v1_context_id),
        metadata_store_pb2.Association(execution_id=me_v1_execution_id, context_id=expt_v1_context_id),
    ]
)

# ── Experiment v2: RandomForestClassifier ─────────────────────────────────
expt_v2_context = metadata_store_pb2.Context()
expt_v2_context.type_id = expt_context_type_id
expt_v2_context.name = 'experiment-rf-v2'
expt_v2_context.properties['model_type'].string_value = 'RandomForestClassifier'
expt_v2_context.properties['note'].string_value       = 'RF n_estimators=100, max_depth=10'
expt_v2_context_id = store.put_contexts([expt_v2_context])[0]

store.put_attributions_and_associations(
    [
        metadata_store_pb2.Attribution(artifact_id=model_v2_artifact_id,   context_id=expt_v2_context_id),
        metadata_store_pb2.Attribution(artifact_id=metrics_v2_artifact_id, context_id=expt_v2_context_id),
    ],
    [
        metadata_store_pb2.Association(execution_id=mt_v2_execution_id, context_id=expt_v2_context_id),
        metadata_store_pb2.Association(execution_id=me_v2_execution_id, context_id=expt_v2_context_id),
    ]
)

print('Experiment v1 context ID:', expt_v1_context_id, '|', expt_v1_context.name)
print('Experiment v2 context ID:', expt_v2_context_id, '|', expt_v2_context.name)

---
## Metadata Store Summary

Before querying lineage, let's confirm everything that was registered.

In [ ]:
all_artifact_types    = {at.id: at.name for at in store.get_artifact_types()}
all_execution_types   = {et.id: et.name for et in store.get_execution_types()}
all_context_types     = {ct.id: ct.name for ct in store.get_context_types()}

print('=== ARTIFACTS ({}) ==='.format(len(store.get_artifacts())))
for a in store.get_artifacts():
    print('  [{:>2}] {:<22} {}'.format(a.id, all_artifact_types.get(a.type_id, '?'), a.uri))

print('\n=== EXECUTIONS ({}) ==='.format(len(store.get_executions())))
for e in store.get_executions():
    print('  [{:>2}] {:<25} state={}'.format(
        e.id,
        all_execution_types.get(e.type_id, '?'),
        e.properties['state'].string_value))

print('\n=== CONTEXTS ({}) ==='.format(len(store.get_contexts())))
for c in store.get_contexts():
    print('  [{:>2}] {:<15} name={}'.format(
        c.id,
        all_context_types.get(c.type_id, '?'),
        c.name))

---
## Lineage Query 1 — Reverse Lineage (Best Model → Raw Data)

Starting from the v2 (RandomForest) model artifact, we walk backwards through the event graph to find the original raw CSV that was used to produce it. In a store with thousands of entries this kind of traversal is the primary debugging tool.

In [ ]:
print('=== LINEAGE QUERY 1: Reverse Lineage — Model v2 → Raw Data ===\n')

# 1. Pick the v2 model
models = store.get_artifacts_by_type('TrainedModel')
target_model = next(m for m in models if m.properties['version'].int_value == 2)
print('Start  : [{}] {} (v{})'.format(
    target_model.id,
    target_model.properties['name'].string_value,
    target_model.properties['version'].int_value))

# 2. Find the execution that produced it (DECLARED_OUTPUT event)
model_events   = store.get_events_by_artifact_ids([target_model.id])
training_exec_id = next(
    e.execution_id for e in model_events
    if e.type == metadata_store_pb2.Event.DECLARED_OUTPUT)
print('  ← produced by execution ID:', training_exec_id)

# 3. Find inputs to that training execution
training_events   = store.get_events_by_execution_ids([training_exec_id])
prep_ids = [e.artifact_id for e in training_events
            if e.type == metadata_store_pb2.Event.DECLARED_INPUT]
prep_artifacts = store.get_artifacts_by_id(prep_ids)
print('  ← training inputs:')
for a in prep_artifacts:
    print('       [{}] {} | {}'.format(a.id, a.type, a.uri))

# 4. For each PreprocessedData artifact, trace back to Feature Engineering
for a in prep_artifacts:
    if a.type == 'PreprocessedData':
        fe_events    = store.get_events_by_artifact_ids([a.id])
        fe_exec_id   = next(
            e.execution_id for e in fe_events
            if e.type == metadata_store_pb2.Event.DECLARED_OUTPUT)
        fe_inp_events = store.get_events_by_execution_ids([fe_exec_id])
        raw_ids      = [e.artifact_id for e in fe_inp_events
                        if e.type == metadata_store_pb2.Event.DECLARED_INPUT]
        raw_artifacts = store.get_artifacts_by_id(raw_ids)
        print('  ← feature engineering (exec {}) inputs:'.format(fe_exec_id))
        for r in raw_artifacts:
            print('       [{}] {} split={} | {}'.format(
                r.id, r.type,
                r.properties.get('split', type('', (), {'string_value': 'n/a'})()).string_value
                    if 'split' in r.properties else 'n/a',
                r.uri))

print('\nFull chain: Model v2 ← Training ← PreprocessedData ← FeatureEng ← Raw CSVs')

## Lineage Query 2 — Forward Lineage (Raw Train Data → All Downstream Artifacts)

Starting from the raw training CSV we walk **forward** through the event graph, discovering every artifact that was derived — directly or transitively — from it.

In [ ]:
print('=== LINEAGE QUERY 2: Forward Lineage — Raw Train CSV → All Downstream ===\n')

raw_train = store.get_artifacts_by_id([train_artifact_id])[0]
print('Start: [{}] {} split={}'.format(
    raw_train.id,
    raw_train.properties['name'].string_value,
    raw_train.properties['split'].string_value))

visited_artifacts  = {raw_train.id}
visited_executions = set()
frontier           = [raw_train.id]

all_execution_types_map = {et.id: et.name for et in store.get_execution_types()}
all_artifact_types_map  = {at.id: at.name for at in store.get_artifact_types()}

while frontier:
    next_frontier = []
    for art_id in frontier:
        events   = store.get_events_by_artifact_ids([art_id])
        exec_ids = [e.execution_id for e in events
                    if e.type == metadata_store_pb2.Event.DECLARED_INPUT]
        for exec_id in exec_ids:
            if exec_id in visited_executions:
                continue
            visited_executions.add(exec_id)
            exec_obj  = store.get_executions_by_id([exec_id])[0]
            exec_name = all_execution_types_map.get(exec_obj.type_id, '?')
            state     = exec_obj.properties['state'].string_value
            print('  Execution [{}]: {} (state={})'.format(exec_id, exec_name, state))

            out_events    = store.get_events_by_execution_ids([exec_id])
            out_artifact_ids = [e.artifact_id for e in out_events
                                if e.type == metadata_store_pb2.Event.DECLARED_OUTPUT]
            out_artifacts = store.get_artifacts_by_id(out_artifact_ids) if out_artifact_ids else []
            for out in out_artifacts:
                type_name = all_artifact_types_map.get(out.type_id, '?')
                print('    -> Output [{}] {} | {}'.format(out.id, type_name, out.uri))
                if out.id not in visited_artifacts:
                    visited_artifacts.add(out.id)
                    next_frontier.append(out.id)
    frontier = next_frontier

print('\nTotal downstream artifacts: {} | Total executions traversed: {}'.format(
    len(visited_artifacts) - 1, len(visited_executions)))

## Lineage Query 3 — Cross-Experiment Comparison

We query the metadata store directly for all `EvaluationMetrics` artifacts and compare the stored scalar properties across experiments — no need to open any files.

In [ ]:
print('=== LINEAGE QUERY 3: Cross-Experiment Metrics Comparison ===\n')

all_metrics = store.get_artifacts_by_type('EvaluationMetrics')

rows = []
for m in all_metrics:
    rows.append({
        'name'    : m.properties['name'].string_value,
        'version' : m.properties['model_version'].int_value,
        'accuracy': round(m.properties['accuracy'].double_value, 4),
        'f1_score': round(m.properties['f1_score'].double_value, 4),
        'uri'     : m.uri,
    })

rows.sort(key=lambda x: x['version'])

print('{:<35} {:>8} {:>10} {:>10}'.format('Name', 'Version', 'Accuracy', 'F1 Score'))
print('-' * 68)
for r in rows:
    print('{:<35} {:>8} {:>10} {:>10}'.format(
        r['name'], r['version'], r['accuracy'], r['f1_score']))

best = max(rows, key=lambda x: x['f1_score'])
print('\nBest model by F1: {} (v{}) — F1={}, Accuracy={}'.format(
    best['name'], best['version'], best['f1_score'], best['accuracy']))

# Load full metrics from file for the best model
print('\nFull metrics for best model (from file):')
with open(best['uri']) as f:
    full = json.load(f)
for k, v in full.items():
    print('  {:<12}: {}'.format(k, v))

---
## Wrap Up

In this lab we built a complete four-step ML pipeline on the **Breast Cancer Wisconsin** dataset and used ML Metadata to record every artifact, execution, and relationship.

**Key differences from the basic MLMD walkthrough:**

| Feature | Basic lab | This lab |
|---------|-----------|----------|
| Storage backend | Fake in-memory | Persistent SQLite (`mlmd.sqlite`) |
| Pipeline steps | 1 (Data Validation only) | 4 (Validation → FE → Training → Evaluation) |
| Artifact types | 3 | 5 (+ PreprocessedData, TrainedModel, EvaluationMetrics) |
| Execution types | 1 | 4 |
| Executions recorded | 1 | 7 (incl. 1 FAILED attempt) |
| Contexts | 1 (Experiment) | 3 (1 Pipeline + 2 Experiment variants) |
| Lineage queries | 1 (reverse only) | 3 (reverse, forward, cross-experiment) |
| Model comparison | None | LogisticRegression vs. RandomForest |

The persistent store and richer lineage graph are the patterns you will find in production TFX pipelines — MLMD scales from this single-notebook demo to pipelines with millions of artifacts by using MySQL or cloud-backed storage instead of SQLite.